# Transfer corrections and the assumption ledger

An experiment rarely measures exactly the estimand a decision needs. `Estimand.transfer_to` says
*which* facets differ and names the licensing assumption for each — but it produces no number.
The operators in `calibrate.transfer` do. Each returns a **`Correction`**: a factor (or offset, or
SE multiplier) together with the `LedgerLine` that records it — the assumption it rests on, the
reading before (`detail["counterfactual"]`) and after (`detail["value"]`) the correction — so no
corrected number can exist without its provenance (rule 4).

`resolve` folds a `TransferPlan` and a sequence of corrections into a `ResolvedTransfer`, and the
`Ledger` enforces the completeness rule (decision D6.5): every differing facet appears in exactly
one facet line that names a typed assumption and carries a counterfactual.

In [ ]:
import numpy as np

from axiom.calibrate import (
    FACET_PREFIX, UNCORRECTED, Correction, CorrectionKind, Ledger, ResolvedTransfer,
    aggregation_level, carryover_window_factor, chord_to_marginal, dose_path_accumulation, resolve,
    variance_reweight, facet_of,
)
from axiom.core import D, Intervention, Outcome, Population, Spec, TimeWindow, Treatment, Unsupported, Verdict
from axiom.estimands import Estimand, Level, Quantity, TransferPlan
from axiom.surface import GeometricCarryover, HillKernel, NoCarryover, Surface, SurfaceSpec

from axiom.display import enable

enable();  # every axiom result renders itself from here on

## Two estimands that differ in window and level

The experiment measured a two-period cumulative contrast at individual level; the decision needs
the eight-period cumulative contrast at cluster level. `transfer_to` returns a `TransferPlan`
whose `differing` facets are `window` and `level`, with one structural ledger line per facet.

In [ ]:
a = Treatment(name="a", dimension=D.currency, unit="USD")
y = Outcome(name="y", dimension=D.outcome)


def estimand(name, *, window, level, kind="contrast", hi=4.0, lo=0.0):
    return Estimand(
        name=name, quantity=Quantity(kind=kind), treatment=a,
        intervention=Intervention(doses={"a": hi}),
        reference=Intervention(doses={"a": lo}) if kind == "contrast" else None,
        outcome=y, population=Population(name="all"), window=window, level=level,
        dimension=D.outcome if kind == "contrast" else D.outcome / D.currency,
    )


source = estimand("experiment", window=TimeWindow(start=0, stop=2, basis="cumulative"), level=Level(unit="individual"))
target = estimand("decision", window=TimeWindow(start=0, stop=8, basis="cumulative"), level=Level(unit="cluster"))
plan: TransferPlan = source.transfer_to(target)
print("status:", plan.status, "| differing:", plan.differing)
for line in plan.ledger_lines:
    print(f"  {line.kind:<14} [{line.assumption.name}] {line.statement}")

## `Ledger.from_plan`: the uncorrected baseline

The plan's lines carry no counterfactual yet — no number exists. `Ledger.from_plan` stamps each
with `detail["counterfactual"] = detail["value"] = UNCORRECTED` and `correction = "none"`: the
counterfactual of an uncorrected facet *is* the source estimate read as-is, and recording that
literally is what makes "we applied no correction here" auditable instead of silent.
`facet_of` reads the facet off a line's `kind` (`FACET_PREFIX + facet`).

In [ ]:
baseline = Ledger.from_plan(plan)
print(FACET_PREFIX, "->", [facet_of(line) for line in baseline.lines])
print("UNCORRECTED =", repr(UNCORRECTED))
print("facets covered:", baseline.facets_covered())
print("complete (explicitly uncorrected counts):", baseline.check_complete(plan).status)
print(baseline.summary())

## `chord_to_marginal`: the chord bias on a known Hill curve

An experiment contrasting doses `d0` and `d1` measures the **chord** `(f(d1) − f(d0)) / (d1 − d0)`;
a `marginal` estimand wants `f'(at)`. On a saturating curve the chord from zero overstates the
marginal at the upper dose — here by more than a factor of two. Both numbers come from the one
`forward()` (`surface.forward` and `surface.marginal`), on the steady-state surface when the spec
declares carryover (0002.19), never a re-implemented Hill.

In [ ]:
K, S, BETA = 2.0, 1.5, 2.0
theta = {"alpha": 0.3, "k_a": K, "s_a": S, "beta_a": BETA, "lam_a": 0.6}
surface = Surface(SurfaceSpec(name="s", treatments=(a,), outcome=y, kernels={"a": HillKernel(reference_dose=K)}))


def hill(x):
    u = (x / K) ** S
    return BETA * u / (1 + u)


def hill_slope(x):
    u = (x / K) ** S
    return BETA * S * (x / K) ** (S - 1) / (K * (1 + u) ** 2)


d0, d1, at = 0.0, 4.0, 4.0
c = chord_to_marginal(surface, theta, "a", d0, d1, at)
assert isinstance(c, Correction)
kind: CorrectionKind = c.kind
print(f"chord over [{d0}, {d1}]: {c.counterfactual:.5f}   marginal at {at}: {c.corrected:.5f}   factor {c.value:.5f}  ({kind})")
print("closed form agrees:", np.isclose(c.counterfactual, (hill(d1) - hill(d0)) / (d1 - d0)), np.isclose(c.corrected, hill_slope(at)))
print(f"reading the chord as the marginal overstates it by {c.counterfactual / c.corrected:.2f}x; chord * factor = {c.counterfactual * c.value:.5f}")
print("facet:", c.facet, "| assumption:", c.ledger_line.assumption.name)
print("degenerate doses ->", type(chord_to_marginal(surface, theta, "a", 1.0, 1.0, 1.0)).__name__)

## Carryover corrections

`carryover_window_factor`: a measurement taken over `w` periods after a dose sees only the share
`Σ_{l<w} w_l` of the carryover mass; the factor `1 / share` scales it up to the steady-state total
(facet `window`). For a geometric kernel with decay `λ` and `max_lag = L`, the share is
`(1 − λ^w) / (1 − λ^L)`. `NoCarryover` gives factor 1 — still a `Correction` with a line, because
"no carryover" is itself the assumption being made.

`dose_path_accumulation`: the effective (carried) dose a path delivered inside its window,
`Σ_t (w ∗ x)_t`, against the raw `Σ_t x_t`; the convolution is `core.causal_convolve` with the
kernel's own weights. Dose carried past the end of the path is what makes the factor fall below
one (facet `intervention`).

In [ ]:
geo = GeometricCarryover(max_lag=6)
lam = 0.7
window = carryover_window_factor(geo, {"lam_a": lam}, 2, treatment="a")
assert isinstance(window, Correction)
print(f"share within 2 periods {window.counterfactual:.5f} (closed form {(1 - lam**2) / (1 - lam**6):.5f}); factor {window.value:.5f}")
print("no carryover:", carryover_window_factor(NoCarryover(), {}, 1, treatment="a").value)

path = np.array([10.0, 10.0, 10.0, 0.0, 0.0, 0.0])
acc = dose_path_accumulation(geo, {"lam_a": lam}, path, treatment="a")
print(f"raw dose {acc.counterfactual:.1f} -> effective {acc.corrected:.4f} inside the path; factor {acc.value:.4f}")
short = dose_path_accumulation(geo, {"lam_a": lam}, path[:3], treatment="a")
print(f"the same pulses with no tail: effective {short.corrected:.4f}; factor {short.value:.4f}")
print("all-zero path ->", type(dose_path_accumulation(geo, {"lam_a": lam}, np.zeros(4), treatment="a")).__name__)

## Level corrections

`variance_reweight` gives the SE at a different aggregation size: `se · sqrt(n_source / n_target)
· sqrt(deff)` with the Kish design effect `deff = 1 + (m − 1)·icc` for the target's clusters of
size `m` (an `se_scale` correction). `aggregation_level` moves between individual, cluster and
aggregate levels: the mean effect *per unit* is invariant under aggregation of an additive effect
(point factor 1, under the `linear_aggregation` assumption) and only the SE changes, through
`variance_reweight`. A difference in interference structure is `Unsupported` — that needs a
declared interference model, not a factor.

In [ ]:
vr = variance_reweight(0.3, 40, 80, icc=0.05, cluster_size=10)
print(f"SE 0.3 for 40 units -> {vr.corrected:.4f} for 80 clustered units (deff {vr.detail['design_effect']}); kind {vr.kind}")
level = aggregation_level(source.level, target.level, cluster_size=25, icc=0.05)
assert isinstance(level, Correction)
print(f"individual -> cluster: point factor {level.detail['point_factor']}, SE factor {level.value:.4f}; facet {level.facet}")
print(type(aggregation_level(Level(unit="individual"), Level(unit="cluster", interference="within_cluster"), cluster_size=25, icc=0.05)).__name__)

## `resolve`: compose the corrections, write the ledger

`resolve` starts from `Ledger.from_plan`; each correction's line **replaces** the plan line for
its facet, so "exactly one line per differing facet" holds by construction and an untouched facet
stays visible as explicitly uncorrected. The composite is `factor` (product of multiplicative
corrections), `offset` (sum of additive) and `se_scale` (product of SE factors).
`ResolvedTransfer.apply(estimate, se)` is `(estimate·factor + offset, se·factor·se_scale)`.
A correction for a facet the plan does not list as differing is a `ValueError` — correcting a facet
that does not differ means the wrong plan or the wrong operator.

In [ ]:
resolved: ResolvedTransfer = resolve(plan, corrections=[window, level])
print("status:", resolved.status, "| licensed:", resolved.licensed)
print(f"factor {resolved.factor:.5f}  offset {resolved.offset}  se_scale {resolved.se_scale:.5f}")
est, se = resolved.apply(1.2, 0.3)
print(f"experiment 1.2 ± 0.3 read as the decision estimand: {est:.4f} ± {se:.4f}")
print("completeness:", resolved.completeness.status, "-", resolved.completeness.reason)
print("round-trips:", Spec.from_json(resolved.to_json()) == resolved)
try:
    resolve(plan, corrections=[c])  # the chord correction serves the intervention facet, which does not differ
except ValueError as e:
    print("ValueError:", str(e)[:80], "...")

A transport verdict (from `identify.transport_verdict`) can accompany the plan: it adds a
`transport` line and, when `identified`, marks the population facet's assumption satisfied; a
`blocked` transport blocks the whole transfer.

In [ ]:
blocked = resolve(plan, transport=Verdict(status="blocked", reason="selection node on the outcome", route="selection_diagram"))
print(blocked.status, "-", blocked.reason)
print([line.kind for line in blocked.ledger.lines])

## The ledger as a table and the completeness check

`to_frame` is one row per line; `append` returns a *new* ledger (Specs are immutable);
`check_complete` returns `identified` when the four-point rule holds and a `blocked` verdict
naming every missing, duplicated, blocked or malformed facet otherwise. It is the gate
`tests/contracts/test_ledger_completeness.py` runs.

In [ ]:
ledger: Ledger = resolved.ledger
frame = ledger.to_frame()
print(frame[["facet", "assumption", "state", "counterfactual", "value", "correction"]].to_string())

In [ ]:
print(ledger.summary())
print()
doubled = ledger.append(ledger.lines[0])
print("appended a duplicate ->", doubled.check_complete(plan).reason)
pruned = Ledger(lines=tuple(ln for ln in ledger.lines if facet_of(ln) != "level"))
print("dropped the level line ->", pruned.check_complete(plan).reason)
print("the original is unchanged:", ledger.facets_covered(), "|", ledger.content_hash()[:12])

## Seeing it

`enable()` at the top of this notebook already made a bare result on the last
line of a cell render itself — a card drawn by `rich`, or the same content as
aligned plain text where `rich` is not installed. `show` does it on demand, for
a result that is not the last thing in its cell.

`axiom.viz` draws the figure this subpackage's results are actually about.

In [ ]:
from axiom.calibrate import variance_reweight
from axiom.display import show
from axiom.viz import corrections

applied = [
    variance_reweight(0.40, n_source=1200, n_target=300),
    variance_reweight(0.31, n_source=300, n_target=900, icc=0.05, cluster_size=20),
]
show(applied[0].ledger_line)
corrections(applied)

The useful question about a calibrated estimate is never *what is the answer* but *which correction moved it*.